In [1]:
#imports
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [25]:
class SVMClassifier1:
    def __init__(self,tol = 1e-8,C = 1.0,no_passes = 10):
        self.tol = tol
        self.C = C
        self.w = None
        self.b = None
        self.max_passes = no_passes
        self.alpha = None
        self.Pos_support_vec = None
        self.neg_support_vec = None
    def pridict(self,x_i):
        return np.dot(x_i, self.w) + self.b
    def K(self,x_i,x_j):
        return np.dot(x_i.T,x_j)
    def __w__(self,x_train,y_train,alpha):
        return np.dot(x_train.T,np.multiply(alpha,y_train))
    def __b__(self,i,j,alpha_i_old,alpha_j_old,E_i,E_j):
        b1 = self.b - E_i - self.y_train[i]*(self.alpha[i] - alpha_i_old)*self.K(self.x_train.iloc[i],self.x_train.iloc[i]) - self.y_train[j]*(self.alpha[j] - alpha_j_old)*self.K(self.x_train.iloc[i],self.x_train.iloc[j])
        b2 = self.b - E_j - self.y_train[i]*(self.alpha[i] - alpha_i_old)*self.K(self.x_train.iloc[i],self.x_train.iloc[j]) - self.y_train[j]*(self.alpha[j] - alpha_j_old)*self.K(self.x_train.iloc[j],self.x_train.iloc[j])
        b = b1 if abs(self.alpha[i]) < self.C else (b2 if abs(self.alpha[j]) < self.C else (b1 + b2 )/2)
        return b
    def validate_KKT(self,x_i,y_i,w,b,alpha_i):# vadidating i_th sample
        margin = y_i*(np.dot(w.T,x_i) + b)
        if alpha_i < self.tol:
            return 1 - self.tol <= margin <= 1 + self.tol
        elif alpha_i > self.C - self.tol:
            return margin <= 1
        elif  0 < alpha_i < self.C - self.tol:
            return abs(margin - 1) <= self.tol
    def fit(self,x_train,y_train):
        self.x_train = x_train
        self.y_train = y_train
        m,n = x_train.shape
        self.alpha = np.zeros(m)
        self.w = np.zeros(n)
        self.b = 0
        passes = 0
        while ( passes < self.max_passes):
            no_changed_alphas = 0
            for i in range(m):
                E_i = self.pridict(self.x_train.iloc[i]) - self.y_train[i]
                #if not ( self.validate_KKT(self.x_train.iloc[i],self.y_train[i],self.w,self.b,self.alpha[i])):
                if ( (self.y_train[i]*E_i < - self.tol and self.alpha[i] < self.C) or (self.y_train[i]*E_i > self.tol and self.alpha[i] < 0)):
                    j = i
                    while j == i:
                        j = np.random.randint(0,m)
                    alpha_i_old = self.alpha[i]
                    alpha_j_old = self.alpha[j]
                    E_j = self.pridict(self.x_train.iloc[j]) - self.y_train[j]
                    L = 0
                    H = self.C
                    if self.y_train[i] != self.y_train[j]:
                        L = max(0, self.alpha[j] - self.alpha[i])
                        H = min(self.C, self.C + self.alpha[j] - self.alpha[i])
                    else:
                        L = max(0, self.alpha[i] + self.alpha[j] - self.C)
                        H = min(self.C,self.alpha[i] + self.alpha[j])
                    if L == H:
                        continue
                    #compute second_der
                    eta = 2*self.K(x_train.iloc[i],x_train.iloc[j]) - self.K(x_train.iloc[i],x_train.iloc[i]) - self.K(x_train.iloc[j],x_train.iloc[j])
                    if eta >= 0:
                        continue
                    self.alpha[j] = alpha_j_old - ( y_train[j]*(E_i - E_j)/eta)
                    self.alpha[j] = min(max(self.alpha[j],L),H)
                    self.alpha[i] = alpha_i_old + y_train[i]*y_train[j]*(alpha_j_old - self.alpha[j])
                    self.b = self.__b__(i,j,alpha_i_old,alpha_j_old,E_i,E_j)
                    no_changed_alphas += 1
                self.w = self.__w__(x_train,y_train,self.alpha)
                if no_changed_alphas <= 0:
                    passes += 1
                
        return self.alpha            

In [26]:
df = pd.read_csv("dataset\Cardiovascular_Disease_Dataset.csv")
df.head()


<>:1: SyntaxWarning: invalid escape sequence '\C'
<>:1: SyntaxWarning: invalid escape sequence '\C'
C:\Users\kumar\AppData\Local\Temp\ipykernel_31784\3084864538.py:1: SyntaxWarning: invalid escape sequence '\C'
  df = pd.read_csv("dataset\Cardiovascular_Disease_Dataset.csv")


,patientid,age,gender,chestpain,restingBP,serumcholestrol,fastingbloodsugar,restingrelectro,maxheartrate,exerciseangia,oldpeak,slope,noofmajorvessels,target
0,103368,53,1,2,171,0,0,1,147,0,5.3,3,3,1
1,119250,40,1,0,94,229,0,1,115,0,3.7,1,1,0
2,119372,49,1,2,133,142,0,0,202,1,5.0,1,0,0
3,132514,43,1,0,138,295,1,1,153,0,3.2,2,2,1
4,146211,31,1,1,199,0,0,2,136,0,5.3,3,2,1


In [27]:
df['target'] = df['target'].replace(0,-1)
df.drop(['patientid'],axis= 1,inplace= True)

In [28]:
x = pd.DataFrame(df.drop(['target'],axis = 1))
x.head()

,age,gender,chestpain,restingBP,serumcholestrol,fastingbloodsugar,restingrelectro,maxheartrate,exerciseangia,oldpeak,slope,noofmajorvessels
0,53,1,2,171,0,0,1,147,0,5.3,3,3
1,40,1,0,94,229,0,1,115,0,3.7,1,1
2,49,1,2,133,142,0,0,202,1,5.0,1,0
3,43,1,0,138,295,1,1,153,0,3.2,2,2
4,31,1,1,199,0,0,2,136,0,5.3,3,2


In [29]:
y = df.target
y

0      1
1     -1
2     -1
3      1
4      1
      ..
995    1
996   -1
997    1
998    1
999   -1
Name: target, Length: 1000, dtype: int64

In [30]:
ratio = 0.8
train_data = int(ratio*x.shape[0])
x_train, x_test = x[:train_data + 1], x[train_data:]
y_train, y_test = y[:train_data + 1], y[train_data:]
x_train


,age,gender,chestpain,restingBP,serumcholestrol,fastingbloodsugar,restingrelectro,maxheartrate,exerciseangia,oldpeak,slope,noofmajorvessels
0,53,1,2,171,0,0,1,147,0,5.3,3,3
1,40,1,0,94,229,0,1,115,0,3.7,1,1
2,49,1,2,133,142,0,0,202,1,5.0,1,0
3,43,1,0,138,295,1,1,153,0,3.2,2,2
4,31,1,1,199,0,0,2,136,0,5.3,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...
796,63,1,3,195,347,1,2,107,0,2.4,2,1
797,22,0,3,200,264,0,0,75,0,0.3,0,0
798,22,0,0,127,330,0,1,193,1,1.4,1,1
799,58,1,1,132,136,0,0,181,1,5.4,0,0


In [31]:

SVM = SVMClassifier1(no_passes= 5)
alpha_optimal = SVM.fit(x_train,y_train)
alpha_optimal


#w_star = SVM.__w__(x_train,y_train,alpha_optimal)


array([ 3.24407279e-04,  1.33496946e-04,  9.67577111e-04,  2.14842692e-03,
        1.97736182e-04,  4.91925155e-05,  1.54366071e-03,  9.64642928e-04,
        2.50558844e-04,  7.44608265e-04,  1.50595629e-04,  0.00000000e+00,
        1.54300218e-03,  1.65641595e-05,  5.29375014e-03,  5.53359734e-04,
        2.18394574e-03,  6.70433606e-03,  2.56252626e-04,  1.46851453e-03,
        0.00000000e+00,  1.01515303e-03,  2.57643863e-04,  3.28051680e-04,
        9.53569101e-04,  8.66286521e-04,  8.01576582e-04,  4.26959486e-04,
        5.27830828e-04,  3.08295768e-06,  0.00000000e+00,  3.17679784e-03,
        1.43307285e-03,  1.68516649e-03,  4.60090903e-05,  4.49632495e-04,
        2.72877311e-03,  4.36477965e-03,  0.00000000e+00,  7.39129419e-04,
        2.81202484e-04,  3.34527115e-03,  1.57524787e-03,  0.00000000e+00,
        2.84000208e-04,  7.71013197e-04,  4.00073120e-03,  3.14549084e-03,
        2.81260253e-04,  3.80518719e-04,  2.91837186e-04,  1.24737812e-05,
        1.67788361e-04,  

In [33]:
m,n = x_train.shape
count = 0
for i in range(800,1000):
  if (SVM.pridict(x.iloc[i])*y[i]) > 0:
    count += 1
count


136